## Part 1: Install & Import

In [1]:
%pip install scikit-learn pandas numpy

In [2]:
import os
import re
import json
import logging
import numpy as np
import pandas as pd
from collections import Counter
from typing import List, Dict, Optional

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Libraries imported successfully.")

Libraries imported successfully.


## Part 2: Mount Google Drive & Konfigurasi Path

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = '/content/drive/MyDrive'
PROJECT_DIR  = os.path.join(BASE_DIR, 'Penalaran Komputer (460, 464)')

# Input dari Tahap 2
PATH_CSV     = os.path.join(PROJECT_DIR, 'Data', 'processed', 'cases.csv')
# Input dari Tahap 3 (query uji)
PATH_QUERIES = os.path.join(PROJECT_DIR, 'Data', 'eval', 'queries.json')

# Output Tahap 4
PATH_RESULTS = os.path.join(PROJECT_DIR, 'Data', 'results')
LOG_PATH     = os.path.join(PROJECT_DIR, 'logs', 'predict.log')

for p in [PATH_RESULTS, os.path.dirname(LOG_PATH)]:
    os.makedirs(p, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_PATH, mode='w', encoding='utf-8'),
        logging.StreamHandler()
    ],
    force=True
)

print(f"CSV Input   : {PATH_CSV}")
print(f"Results Dir : {PATH_RESULTS}")

Mounted at /content/drive
CSV Input   : /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/processed/cases.csv
Results Dir : /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/results


## Part 3: Load Data & Rebuild TF-IDF + SVM dari Tahap 3

In [4]:
df = pd.read_csv(PATH_CSV)
print(f"Total kasus dimuat: {len(df)}")

# Rebuild text_feature
df['text_feature'] = (
    df['full_text'].fillna('') + ' ' + df['ringkasan_fakta'].fillna('')
).str.strip()

# Rebuild label
def simplify_label(pasal):
    if pd.isna(pasal) or str(pasal).strip() == '':
        return 'tidak diketahui'
    match = re.search(r'pasal\s*(\d+)', str(pasal), re.IGNORECASE)
    return f"pasal_{match.group(1)}" if match else 'lainnya'

df['label_simplified'] = df['pasal'].apply(simplify_label)

class_counts = df['label_simplified'].value_counts()
df = df[df['label_simplified'].isin(class_counts[class_counts >= 2].index)].reset_index(drop=True)

# Rebuild TF-IDF (Menggunakan df yang sudah bersih agar dimensi matriks sinkron)
tfidf = TfidfVectorizer(min_df=2, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)
texts       = df['text_feature'].tolist()
case_ids    = df['case_id'].tolist()
tfidf_matrix = tfidf.fit_transform(texts)

# Rebuild SVM
le = LabelEncoder()
labels_encoded = le.fit_transform(df['label_simplified'])
indices = np.arange(len(df))
idx_train, idx_test = train_test_split(
    indices, test_size=0.2, random_state=42,
    stratify=labels_encoded if len(np.unique(labels_encoded)) > 1 else None
)
X_train = tfidf_matrix[idx_train]
y_train = labels_encoded[idx_train]

svm_model = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_model.fit(X_train, y_train)

print(f"Total kasus setelah filter kelas langka: {len(df)}")
print("TF-IDF + SVM berhasil di-rebuild.")
logging.info("Model rebuilt from CSV.")

Total kasus dimuat: 50


2026-06-27 05:56:14,255 - INFO - Model rebuilt from CSV.


Total kasus setelah filter kelas langka: 42
TF-IDF + SVM berhasil di-rebuild.


## Part 4: Ekstraksi Solusi (Amar Putusan)

In [5]:
def extract_amar_putusan(text: str) -> str:
    """
    Ekstrak amar putusan dari full_text.
    Amar putusan biasanya dimulai dengan kata:
    'mengadili', 'memutuskan', 'amar putusan', 'menyatakan'
    """
    if not text or len(str(text)) < 10:
        return 'amar putusan tidak tersedia'

    text = str(text)

    # Cari section amar putusan
    patterns = [
        r'(mengadili[\s\S]{0,1500})',
        r'(memutuskan[\s\S]{0,1500})',
        r'(amar putusan[\s\S]{0,1500})',
        r'(menyatakan terdakwa[\s\S]{0,800})',
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            raw = match.group(1).strip()
            # Bersihkan: ambil 500 karakter pertama, hapus newline berlebih
            cleaned = re.sub(r'\s+', ' ', raw)[:500].strip()
            return cleaned

    # Fallback: ambil 300 karakter terakhir teks (biasanya berisi amar)
    fallback = re.sub(r'\s+', ' ', text[-300:]).strip()
    return fallback if fallback else 'amar putusan tidak tersedia'


# Bangun case_solutions: {case_id: amar_putusan}
case_solutions = {}
for _, row in df.iterrows():
    cid   = row['case_id']
    amar  = extract_amar_putusan(row.get('full_text', ''))
    case_solutions[cid] = amar

# Simpan ke JSON
solutions_path = os.path.join(PATH_RESULTS, 'case_solutions.json')
with open(solutions_path, 'w', encoding='utf-8') as f:
    json.dump(case_solutions, f, ensure_ascii=False, indent=2)

print(f"Berhasil mengekstrak solusi untuk {len(case_solutions)} kasus.")
print(f"Disimpan ke: {solutions_path}")

# Preview
sample_id = list(case_solutions.keys())[0]
print(f"\nContoh solusi [{sample_id}]:\n{case_solutions[sample_id][:300]}...")
logging.info(f"Extracted solutions for {len(case_solutions)} cases.")

2026-06-27 05:56:15,000 - INFO - Extracted solutions for 42 cases.


Berhasil mengekstrak solusi untuk 42 kasus.
Disimpan ke: /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/results/case_solutions.json

Contoh solusi [case_005]:
mengadili dan memutus perkara perkara pidana pada pengadilan tingkat pertama dengan acara pemeriksaan biasa telah menjatuhkan putusan sebagai berikut di bawah ini dalam perkara terdakwa nama lengkap porman boru simanjuntak tempat lahir tebing tinggi sumut umur tanggal lahir 46 tahun 06 desember 1966...


## Part 5: Rebuild Fungsi Retrieval dari Tahap 3

In [6]:
def retrieve(query: str, k: int = 5) -> List[Dict]:
    """
    Retrieve top-k kasus paling mirip dengan query via cosine similarity.
    Returns list of dict: {case_id, similarity_score, pasal, ringkasan_fakta}
    """
    query_vec   = tfidf.transform([query])
    sim_scores  = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_k_idx   = np.argsort(sim_scores)[::-1][:k]

    results = []
    for idx in top_k_idx:
        row = df.iloc[idx]
        results.append({
            'case_id'         : row['case_id'],
            'similarity_score': round(float(sim_scores[idx]), 4),
            'pasal'           : str(row.get('pasal', '')),
            'ringkasan_fakta' : str(row.get('ringkasan_fakta', ''))[:300],
        })
    return results

print("Fungsi retrieve() siap.")

Fungsi retrieve() siap.


## Part 6: Fungsi predict_outcome() — Majority Vote & Weighted Similarity

In [7]:
def predict_outcome(
    query: str,
    k: int = 5,
    method: str = 'weighted'   # 'majority' atau 'weighted'
) -> Dict:
    """
    Prediksi solusi (amar putusan) untuk kasus baru.

    Args:
        query  : teks deskripsi kasus baru
        k      : jumlah kasus acuan
        method : 'majority' = voting murni
                 'weighted' = bobot berdasarkan similarity score

    Returns:
        dict berisi predicted_solution, top_k_case_ids, method, scores
    """
    top_k = retrieve(query, k=k)

    if not top_k:
        return {
            'predicted_solution': 'tidak dapat diprediksi (retrieval kosong)',
            'top_k_case_ids'    : [],
            'method'            : method,
            'scores'            : {}
        }

    top_k_ids = [r['case_id'] for r in top_k]
    solutions = [case_solutions.get(cid, 'tidak tersedia') for cid in top_k_ids]
    sim_scores = [r['similarity_score'] for r in top_k]

    if method == 'majority':
        # ── Majority Vote ──────────────────────────────────────────────
        # Hitung frekuensi kemunculan tiap solusi unik
        vote_counter = Counter(solutions)
        predicted_solution = vote_counter.most_common(1)[0][0]
        score_detail = dict(vote_counter)

    else:
        # ── Weighted Similarity ────────────────────────────────────────
        # Solusi unik → akumulasi bobot similarity
        weighted_scores: Dict[str, float] = {}
        for sol, sim in zip(solutions, sim_scores):
            weighted_scores[sol] = weighted_scores.get(sol, 0.0) + sim

        # Pilih solusi dengan total bobot tertinggi
        predicted_solution = max(weighted_scores, key=weighted_scores.get)
        score_detail = {k: round(v, 4) for k, v in weighted_scores.items()}

    logging.info(
        f"predict_outcome [{method}]: top_k={top_k_ids}, "
        f"predicted='{predicted_solution[:80]}'"
    )

    return {
        'predicted_solution': predicted_solution,
        'top_k_case_ids'    : top_k_ids,
        'similarity_scores' : sim_scores,
        'method'            : method,
        'score_detail'      : score_detail
    }

print("Fungsi predict_outcome() siap.")

Fungsi predict_outcome() siap.


## Part 7: Demo Manual — 5 Kasus Baru

In [8]:
# Ambil 5 kasus dari test set sebagai simulasi "kasus baru"
# Query = potongan teks awal (bukan full_text agar tidak identik 100%)
demo_cases = []
for idx in idx_test[:5]:
    row = df.iloc[idx]
    demo_cases.append({
        'case_id'          : row['case_id'],
        'query'            : str(row['text_feature'])[:400],   # Simulasi kasus baru
        'actual_solution'  : case_solutions.get(row['case_id'], 'tidak tersedia'),
        'actual_pasal'     : str(row.get('pasal', ''))
    })

print("=" * 70)
print("DEMO MANUAL — PREDIKSI SOLUSI UNTUK 5 KASUS BARU")
print("=" * 70)

for i, case in enumerate(demo_cases, 1):
    result_w = predict_outcome(case['query'], k=5, method='weighted')
    result_m = predict_outcome(case['query'], k=5, method='majority')

    print(f"\n{'─'*70}")
    print(f"KASUS {i}: {case['case_id']}")
    print(f"Query     : {case['query'][:150]}...")
    print(f"Pasal Asli: {case['actual_pasal']}")
    print(f"\nTop-5 Kasus Acuan (Weighted):")
    for j, (cid, sim) in enumerate(
        zip(result_w['top_k_case_ids'], result_w['similarity_scores']), 1
    ):
        print(f"  {j}. {cid} | sim={sim:.4f}")
    print(f"\n[Weighted] Solusi Prediksi  : {result_w['predicted_solution'][:200]}")
    print(f"[Majority] Solusi Prediksi  : {result_m['predicted_solution'][:200]}")
    print(f"Solusi Aktual               : {case['actual_solution'][:200]}")

    # Cek kesesuaian kasar (apakah ada overlap kata kunci)
    keywords = ['bersalah', 'bebas', 'pidana', 'penjara', 'kurungan', 'denda']
    pred_kw  = [kw for kw in keywords if kw in result_w['predicted_solution'].lower()]
    act_kw   = [kw for kw in keywords if kw in case['actual_solution'].lower()]
    overlap  = set(pred_kw) & set(act_kw)
    print(f"\nKata kunci prediksi : {pred_kw}")
    print(f"Kata kunci aktual   : {act_kw}")
    print(f"Overlap kunci       : {list(overlap)} {'✓ Sesuai' if overlap else '✗ Berbeda'}")

2026-06-27 05:56:15,077 - INFO - predict_outcome [weighted]: top_k=['case_026', 'case_014', 'case_032', 'case_025', 'case_030'], predicted='mengadili perkara perdata agama pada tingkat pertama dalamsidang majelis hakim y'
2026-06-27 05:56:15,084 - INFO - predict_outcome [majority]: top_k=['case_026', 'case_014', 'case_032', 'case_025', 'case_030'], predicted='mengadili perkara perdata agama pada tingkat pertama dalamsidang majelis hakim y'
2026-06-27 05:56:15,092 - INFO - predict_outcome [weighted]: top_k=['case_036', 'case_018', 'case_034', 'case_038', 'case_040'], predicted='mengadili perkara perkara pidana pada pengadilan tingkat pertama dengan acara pe'
2026-06-27 05:56:15,099 - INFO - predict_outcome [majority]: top_k=['case_036', 'case_018', 'case_034', 'case_038', 'case_040'], predicted='mengadili perkara perkara pidana pada pengadilan tingkat pertama dengan acara pe'
2026-06-27 05:56:15,108 - INFO - predict_outcome [weighted]: top_k=['case_031', 'case_001', 'case_035', 'case_02

DEMO MANUAL — PREDIKSI SOLUSI UNTUK 5 KASUS BARU

──────────────────────────────────────────────────────────────────────
KASUS 1: case_026
Query     : p u t u s a nnomor 119 pdt g 2025 pa rmbdemi keadilan berdasarkan ketuhanan yang maha esapengadilan agama rumbiamemeriksa dan mengadili perkara perdat...
Pasal Asli: pasal 49, 3 tahun 2006

Top-5 Kasus Acuan (Weighted):
  1. case_026 | sim=0.2613
  2. case_014 | sim=0.1812
  3. case_032 | sim=0.1571
  4. case_025 | sim=0.1439
  5. case_030 | sim=0.0953

[Weighted] Solusi Prediksi  : mengadili perkara perdata agama pada tingkat pertama dalamsidang majelis hakim yang dilaksanakan secara elektronik telah menjatuhkanputusan dalam perkara cerai gugat antara penggugat nik tempat dan ta
[Majority] Solusi Prediksi  : mengadili perkara perdata agama pada tingkat pertama dalamsidang majelis hakim yang dilaksanakan secara elektronik telah menjatuhkanputusan dalam perkara cerai gugat antara penggugat nik tempat dan ta
Solusi Aktual               : m

## Part 8: Simpan predictions.csv

In [9]:
# Jalankan predict_outcome untuk SEMUA query di queries.json (dari Tahap 3)
# Fallback: gunakan demo_cases jika queries.json belum ada

if os.path.exists(PATH_QUERIES):
    with open(PATH_QUERIES, 'r', encoding='utf-8') as f:
        eval_queries = json.load(f)
    print(f"Loaded {len(eval_queries)} queries dari {PATH_QUERIES}")
else:
    # Buat dari test set jika file belum ada
    eval_queries = []
    for idx in idx_test:
        row = df.iloc[idx]
        eval_queries.append({
            'query_id'         : f"query_{len(eval_queries)+1:03d}",
            'query_text'       : str(row['text_feature'])[:500],
            'ground_truth_case': row['case_id'],
            'ground_truth_label': row['label_simplified']
        })
    print(f"queries.json tidak ditemukan. Dibuat {len(eval_queries)} query dari test set.")


# Jalankan prediksi untuk setiap query
rows = []
for q in eval_queries:
    res_w = predict_outcome(q['query_text'], k=5, method='weighted')
    res_m = predict_outcome(q['query_text'], k=5, method='majority')

    rows.append({
        'query_id'                  : q['query_id'],
        'ground_truth_case'         : q.get('ground_truth_case', ''),
        'top_5_case_ids'            : ', '.join(res_w['top_k_case_ids']),
        'similarity_scores'         : ', '.join([str(s) for s in res_w['similarity_scores']]),
        'predicted_solution_weighted': res_w['predicted_solution'][:300],
        'predicted_solution_majority': res_m['predicted_solution'][:300],
    })

predictions_df = pd.DataFrame(rows)

# Simpan
pred_path = os.path.join(PATH_RESULTS, 'predictions.csv')
predictions_df.to_csv(pred_path, index=False, encoding='utf-8')

predictions_df[['query_id', 'ground_truth_case', 'top_5_case_ids',
                'similarity_scores', 'predicted_solution_weighted']]\
    .rename(columns={'predicted_solution_weighted': 'predicted_solution'})\
    .to_csv(os.path.join(PATH_RESULTS, 'svm_predictions.csv'), index=False, encoding='utf-8')

predictions_df[['query_id', 'ground_truth_case', 'top_5_case_ids',
                'similarity_scores', 'predicted_solution_majority']]\
    .rename(columns={'predicted_solution_majority': 'predicted_solution'})\
    .to_csv(os.path.join(PATH_RESULTS, 'nb_predictions.csv'), index=False, encoding='utf-8')

print(f"✅ Saved svm_predictions.csv & nb_predictions.csv ke: {PATH_RESULTS}")

2026-06-27 05:56:15,850 - INFO - predict_outcome [weighted]: top_k=['case_014', 'case_026', 'case_032', 'case_025', 'case_030'], predicted='mengadili perkaratertentu pada tingkat pertama dalam persidangan majelis telah m'
2026-06-27 05:56:15,859 - INFO - predict_outcome [majority]: top_k=['case_014', 'case_026', 'case_032', 'case_025', 'case_030'], predicted='mengadili perkaratertentu pada tingkat pertama dalam persidangan majelis telah m'
2026-06-27 05:56:15,869 - INFO - predict_outcome [weighted]: top_k=['case_040', 'case_034', 'case_038', 'case_018', 'case_006'], predicted='mengadili perkara perkara pidana dengan acara pemeriksaan biasa pada peradilan t'
2026-06-27 05:56:15,879 - INFO - predict_outcome [majority]: top_k=['case_040', 'case_034', 'case_038', 'case_018', 'case_006'], predicted='mengadili perkara perkara pidana dengan acara pemeriksaan biasa pada peradilan t'
2026-06-27 05:56:15,889 - INFO - predict_outcome [weighted]: top_k=['case_031', 'case_035', 'case_001', 'case_02

Loaded 10 queries dari /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/eval/queries.json
✅ Saved svm_predictions.csv & nb_predictions.csv ke: /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/results


## Part 9: Evaluasi Prediksi Solusi

In [10]:
# Hitung seberapa sering ground-truth case masuk top-5
def eval_predictions(predictions_df: pd.DataFrame) -> Dict:
    if 'ground_truth_case' not in predictions_df.columns:
        return {}

    total = len(predictions_df)
    hits  = 0
    for _, row in predictions_df.iterrows():
        top5 = [x.strip() for x in str(row['top_5_case_ids']).split(',')]
        if row['ground_truth_case'] in top5:
            hits += 1

    # Evaluasi overlap kata kunci amar putusan
    keywords  = ['bersalah', 'bebas', 'pidana', 'penjara', 'kurungan', 'denda', 'terbukti']
    kw_matches = 0
    for _, row in predictions_df.iterrows():
        gt_case  = row.get('ground_truth_case', '')
        act_sol  = case_solutions.get(gt_case, '')
        pred_sol = str(row.get('predicted_solution_weighted', ''))
        act_kw   = set(kw for kw in keywords if kw in act_sol.lower())
        pred_kw  = set(kw for kw in keywords if kw in pred_sol.lower())
        if act_kw & pred_kw:
            kw_matches += 1

    return {
        'total_queries'         : total,
        'retrieval_hits_at_5'   : hits,
        'retrieval_accuracy_at_5': round(hits / total, 4) if total else 0,
        'keyword_match_count'   : kw_matches,
        'keyword_match_rate'    : round(kw_matches / total, 4) if total else 0,
    }

eval_result = eval_predictions(predictions_df)
print("=== Evaluasi Prediksi Solusi ===")
for k, v in eval_result.items():
    print(f"  {k}: {v}")

# Simpan ke CSV
pred_metrics_path = os.path.join(PATH_RESULTS, 'prediction_metrics.csv')
pd.DataFrame([eval_result]).to_csv(pred_metrics_path, index=False)
print(f"\nMetrik prediksi disimpan ke: {pred_metrics_path}")
logging.info(f"Prediction eval: {eval_result}")

=== Evaluasi Prediksi Solusi ===
  total_queries: 10
  retrieval_hits_at_5: 10
  retrieval_accuracy_at_5: 1.0
  keyword_match_count: 7
  keyword_match_rate: 0.7


2026-06-27 05:56:19,000 - INFO - Prediction eval: {'total_queries': 10, 'retrieval_hits_at_5': 10, 'retrieval_accuracy_at_5': 1.0, 'keyword_match_count': 7, 'keyword_match_rate': 0.7}



Metrik prediksi disimpan ke: /content/drive/MyDrive/Penalaran Komputer (460, 464)/Data/results/prediction_metrics.csv
